# AQNet — Three-Tier PM2.5 Research Pipeline

Runs the **AQNet** offline research model of the [Shared Skies Initiative](https://sharedskiesinitiative.org/real-time-map)
end to end on a fresh Colab runtime. AQNet is a research track — it does **not** feed the live map.

**The three tiers**

1. **Tier 1 — tabular:** GBM ensemble (LightGBM / XGBoost / CatBoost / Random Forest) with simplex-blended
   out-of-fold predictions and LightGBM quantile heads, leave-one-sensor-out (LOSO) cross-validated.
2. **Tier 2 — deep:** the existing `FusionUNet` (per-source spatial-attention fusion → U-Net) on a gridded
   0.1° Texas stack, extended with GEOS-CF and MERRA-2 channels when available.
3. **Tier 3 — fusion:** residual kriging of Tier-1 errors + a stacked ridge meta-learner over strictly
   out-of-fold predictions + split-conformal prediction intervals.

Validation: LOSO, spatial-block, and temporal CV; interpolation and raw-CTM baselines; and **external
EPA AQS FRM/FEM monitors that the models never train on**.

**Before you start:** switch to a GPU runtime (`Runtime → Change runtime type → GPU`). The tabular and
validation stages run fine on CPU, but the deep stage needs a GPU to finish in hours instead of days.

No performance numbers are promised anywhere in this notebook — every metric you see at the end is
computed by the run itself.

In [ ]:
# 1) Clone the repo (~400 MB of data included — this takes a minute or two)
import os

if os.path.basename(os.getcwd()) != "shared-skies-initiative":
    if not os.path.isdir("shared-skies-initiative"):
        print("Cloning shared-skies-initiative (~400 MB)...")
        !git clone --depth 1 https://github.com/saketh-c/shared-skies-initiative
    %cd shared-skies-initiative
print("Working directory:", os.getcwd())

In [ ]:
# 2) Install AQNet dependencies (torch ships with Colab; the rest is quick)
%pip install -q -r research/aqnet/requirements.txt

## 3) Optional: NASA Earthdata login (enables MERRA-2 aerosol species + PBLH)

MERRA-2 requires a **free** NASA Earthdata account:

1. Register at <https://urs.earthdata.nasa.gov/users/new> (takes ~2 minutes).
2. Run the cell below and enter your username and password when prompted.

**This step is entirely skippable.** If you skip it (or the login fails), the pipeline runs with
`--skip-merra2` automatically: the `merra2_*` feature columns are simply NaN and every downstream
stage degrades gracefully.

In [ ]:
# 3) Optional Earthdata login — safe to skip
MERRA2_FLAG = "--skip-merra2"
try:
    import earthaccess
    auth = earthaccess.login(strategy="interactive", persist=True)
    if getattr(auth, "authenticated", False):
        MERRA2_FLAG = ""
        print("Earthdata login OK — the data stage will fetch MERRA-2.")
    else:
        print("Not authenticated — continuing with --skip-merra2 (merra2_* features -> NaN).")
except Exception as e:
    print(f"Earthdata login skipped ({e}) — continuing with --skip-merra2.")
print("MERRA2_FLAG =", repr(MERRA2_FLAG))

## 4) Smoke test first (recommended)

`--quick` runs **every** stage on a small slice — a 3-month 2024 window, a coarser 0.2° grid, 4 LOSO
folds, and 3 U-Net epochs — so you can validate the whole plumbing (downloads, feature build, all three
tiers, validation artifacts) before committing to the multi-hour full run. Expect roughly 5–15 minutes,
dominated by the external downloads.

Artifacts land in `research/aqnet/artifacts/` and are **overwritten by the full run below**, so treat
quick outputs as plumbing checks, not results.

In [ ]:
# 4) Quick end-to-end smoke test (~5-15 min, mostly download time)
MERRA2_FLAG = globals().get("MERRA2_FLAG", "--skip-merra2")
!python research/aqnet/pipeline_colab.py all --quick {MERRA2_FLAG}

## 5) Full run, stage by stage

Rough expectations on a Colab GPU runtime (network speed and library versions move these a lot):

| Stage | What it does | Rough runtime |
|---|---|---|
| `data` | EPA AQS zips + GEOS-CF OPeNDAP (+ MERRA-2 if logged in) | minutes to ~1 h (network-bound; MERRA-2 is the slow one) |
| `features` | training frame + neighbor features + frozen CV folds | minutes |
| `tabular` | Tier-1 LOSO CV (4 models x 10 folds) + quantile heads | tens of minutes |
| `deep` | extended grid stack + FusionUNet training | **hours** (GPU strongly recommended) |
| `fuse` | residual kriging + meta-learner + conformal calibration | tens of minutes (per-day kriging) |
| `validate` | spatial/temporal CV retrains, baselines, external AQS, SHAP | tens of minutes to ~1 h |

Each stage is restartable — if the runtime disconnects, re-run from the stage that failed. Stages print
what they skipped and why.

In [ ]:
# 5a) data — fetch external open datasets
MERRA2_FLAG = globals().get("MERRA2_FLAG", "--skip-merra2")
!python research/aqnet/pipeline_colab.py data {MERRA2_FLAG}

In [ ]:
# 5b) features — training frame + frozen CV folds
!python research/aqnet/pipeline_colab.py features

In [ ]:
# 5c) tabular — Tier 1: GBM ensemble LOSO CV + quantile heads
!python research/aqnet/pipeline_colab.py tabular

In [ ]:
# 5d) deep — Tier 2: FusionUNet on the extended grid stack (hours on GPU)
!python research/aqnet/pipeline_colab.py deep

In [ ]:
# 5e) fuse — Tier 3: residual kriging + stacked meta-learner + conformal
!python research/aqnet/pipeline_colab.py fuse

In [ ]:
# 5f) validate — all metrics artifacts + SUMMARY.md
!python research/aqnet/pipeline_colab.py validate

## 6) Results

The cells below only display what the run produced — each `metrics_*.json` as a table, the SHAP summary
plot, and the auto-generated `SUMMARY.md`. Missing files mean the corresponding stage was skipped
(the stage logs above say why).

In [ ]:
# 6a) Every metrics_*.json as a table
import glob, json, os
import pandas as pd
from IPython.display import display, Markdown

ART = "research/aqnet/artifacts"
found = sorted(glob.glob(os.path.join(ART, "metrics_*.json")))
if not found:
    print("No metrics_*.json artifacts yet — run the validate stage first.")
for path in found:
    with open(path) as f:
        obj = json.load(f)
    display(Markdown(f"### `{os.path.basename(path)}`"))
    table = pd.json_normalize(obj, sep=".").T
    table.columns = ["value"]
    display(table)

In [ ]:
# 6b) SHAP summary plot (Tier-1 LightGBM)
import os
from IPython.display import Image, display

p = "research/aqnet/artifacts/shap_summary.png"
if os.path.exists(p):
    display(Image(p))
else:
    print("shap_summary.png not present — the interpretability step was "
          "skipped (see the validate-stage log).")

In [ ]:
# 6c) Auto-generated run summary
import os
from IPython.display import Markdown, display

p = "research/aqnet/artifacts/SUMMARY.md"
if os.path.exists(p):
    with open(p) as f:
        display(Markdown(f.read()))
else:
    print("SUMMARY.md not present — run the validate stage first.")